Universidad Torcuato Di Tella

Licenciatura en Tecnología Digital\
**Tecnología Digital VI: Inteligencia Artificial**

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Importamos librerias

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
import zipfile
import os
from PIL import Image

Para esta prueba vamos a utilizar el dataset [SIPAKMED](https://www.cs.uoi.gr/~marina/sipakmed.html). Se trata de un dataset de imágenes de microscopía a partir de muestras de PAPs. Tiene 5 clases, donde la anotación está compuesta por las coordenadas que conforman el contorno de las células. Para esta clase se convirtieron las anotaciones en máscaras. En particular vamos a trabajar con una sola de las 5 clases, que es la que contiene las células Metaplásticas.

In [ ]:
with zipfile.ZipFile("/content/drive/MyDrive/sipakmed_metaplastic.zip", 'r') as zip_ref:
    zip_ref.extractall('./')

Definimos el Dataset

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.transform = transform

    def __len__(self):
        return len(os.listdir(self.images_dir))

    def __getitem__(self, idx):
        img_path = os.path.join(self.images_dir, os.listdir(self.images_dir)[idx])
        mask_path = os.path.join(self.masks_dir, os.listdir(self.masks_dir)[idx])
        image = Image.open(img_path)
        mask = Image.open(mask_path)
        if self.transform is not None:
            image = self.transform(image)
            mask = self.transform(mask)
        return image, mask

In [ ]:
# Calcular el relleno necesario
padding_left = padding_right = 0
padding_top = padding_bottom = (2048 - 1536) // 2 #La imagen originalmente no es cuadrada

# Definir las transformaciones
transform = transforms.Compose([
    transforms.Pad((padding_left, padding_top, padding_right, padding_bottom), fill=0), # Rellenamos con negro
    transforms.esize((572, 572))R,
    transforms.ToTensor()
])

In [ ]:
dataset = CustomDataset('sipakmed_metaplastic/images', 'sipakmed_metaplastic/masks', transform=transform)

# Split the dataset into train and test sets
train_val_dataset, test_dataset = train_test_split(dataset, test_size=0.2, random_state=42)

# Split the train set into train and validation sets
train_dataset, val_dataset = train_test_split(train_val_dataset, test_size=0.2, random_state=42)

# Define the loaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)


Observemos algunas imágenes y sus máscaras

In [ ]:
dataiter = iter(train_loader)
images, labels = next(dataiter)

plt.figure(figsize=(20, 10))
for idx in range(4):
    plt.subplot(2, 4, idx + 1)
    plt.imshow(images[idx].permute(1, 2, 0))
    plt.title("Original Image")
    plt.axis("off")

    plt.subplot(2, 4, idx + 5)
    plt.imshow(labels[idx].permute(1, 2, 0))
    plt.title("Ground truth")
    plt.axis("off")


plt.show()


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Arquitectura de U-Net

Definimos los Convolution Blocks individuales. Cada uno de ellos consiste en dos convoluciones 3x3 seguidas de una activación ReLU.

In [ ]:
class conv_block(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(out_c, out_c, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
    def forward(self, inputs):
        x = self.conv1(inputs)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.relu(x)
        return x

Definimos los Encoding Blocks. Cada uno de ellos consiste en un Convolution Block seguido de un Max Pooling 2x2. Notemos que despues de cada encoding block la cantidad de canales se duplica y la resolución espacial se reduce a la mitad.

In [ ]:
class encoder_block(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = conv_block(in_c, out_c)
        self.pool = nn.MaxPool2d((2, 2))
    def forward(self, inputs):
        x = self.conv(inputs)
        p = self.pool(x)
        return x, p

Definimos ahora los Decoding blocks. Observemos que se hace un upsampling seguido de una concatenación con el feature map del encoding block correspondiente. Luego se aplica un Convolution Block.

In [ ]:
class decoder_block(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_c, out_c, kernel_size=2, stride=2, padding=0)
        self.conv = conv_block(out_c+out_c, out_c)
    def forward(self, inputs, skip):
        x = self.up(inputs)
        # Ajuste de dimensiones
        diffY = skip.size()[2] - x.size()[2]
        diffX = skip.size()[3] - x.size()[3]

        x = F.pad(x, [diffX // 2, diffX - diffX // 2,
                      diffY // 2, diffY - diffY // 2])
        x = torch.cat([x, skip], axis=1)
        x = self.conv(x)
        return x

Ahora si, definimos la arquitectura completa de U-Net.

In [ ]:
class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        """ Encoder """
        self.e1 = encoder_block(3, 64)
        self.e2 = encoder_block(64, 128)
        self.e3 = encoder_block(128, 256)
        self.e4 = encoder_block(256, 512)
        """ Bottleneck """
        self.b = conv_block(512, 1024)
        """ Decoder """
        self.d1 = decoder_block(1024, 512)
        self.d2 = decoder_block(512, 256)
        self.d3 = decoder_block(256, 128)
        self.d4 = decoder_block(128, 64)
        """ Classifier """
        self.outputs = nn.Conv2d(64, 1, kernel_size=1, padding=0)
    def forward(self, inputs):
        """ Encoder """
        s1, p1 = self.e1(inputs)
        s2, p2 = self.e2(p1)
        s3, p3 = self.e3(p2)
        s4, p4 = self.e4(p3)
        """ Bottleneck """
        b = self.b(p4)
        """ Decoder """
        d1 = self.d1(b, s4)
        d2 = self.d2(d1, s3)
        d3 = self.d3(d2, s2)
        d4 = self.d4(d3, s1)
        """ Classifier """
        outputs = self.outputs(d4)
        return outputs

model = UNet().to(device)

Definimos función de pérdida y optimizador

In [ ]:
criterion = nn.BCELoss().to(device) # Funcion de pérdida
# "Creates a criterion that measures the Binary Cross Entropy between the target and the input"
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

In [ ]:
num_epochs = 10

for epoch in range(num_epochs):
  train_loss = []
  val_loss = []

  for data in train_loader:
    imgs, labels = data
    imgs = imgs.to(device)
    labels = labels.to(device)
    outputs = model(imgs)
    outputs = torch.sigmoid(outputs)
    loss = criterion(outputs, labels) # Comparar la segmentacion generada con el ground truth
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    train_loss.append(loss.item() * imgs.size(0))


  with torch.no_grad():
    for data_val in val_loader:
      imgs_val, labels_val = data_val
      imgs_val = imgs_val.to(device)
      labels_val = labels_val.to(device)
      outputs_val = model(imgs_val)
      outputs_val = torch.sigmoid(outputs_val)
      loss_val = criterion(outputs_val, labels_val) # Comparar la imagen reconstruida con la original
      val_loss.append(loss_val.item() * imgs_val.size(0))




  print(f'Epoch: {epoch+1}, Train Loss: {np.mean(train_loss):.4f}, Val Loss: {np.mean(val_loss):.4f}')